In [ ]:

# ============================================================
# REGRAS DA LÓGICA
# ============================================================
# ~   : negação
# &   : conjunção
# |   : disjunção
# ->  : implicação
# <-> : bicondicional
# Toda operação binária deve estar entre parênteses
# ============================================================

import pandas as pd 


def remover_parenteses_externos(expr):
    # Remove parênteses redundantes externos
    while True:
        if len(expr) < 2 or expr[0] != "(" or expr[-1] != ")": # Garante que a expressão tem pelo menos dois caracteres 
                                                                # e começa e termina com parênteses
            return expr

        nivel = 0 # Nível é um contador de parênteses abertos
        pode_remover = True

        for i in range(len(expr)):
            if expr[i] == "(": 
                nivel += 1 # Incrementa o nível ao encontrar um parêntese de abertura
            elif expr[i] == ")":
                nivel -= 1 # Decrementa o nível ao encontrar um parêntese de fechamento

            if nivel == 0 and i != len(expr) - 1: # Se o nível zerar antes do último caractere da string, 
                                                # significa que os parênteses externos não incluem a expressão inteira.
                                                # Portanto, não são redundantes e não podem ser removidos.
                pode_remover = False
                break

        if pode_remover:
            expr = expr[1:-1] # Faz o slicing para tirar o primeiro e o último dígito, que são os parênteses redundantes

        else:
            return expr
            # Se a expressão é retornada intacta, significa que a função identificou que os parênteses das 
            # pontas são obrigatórios para manter a ordem matemática expressão, ou seja, eles não são redundantes.

def encontrar_relacoes(expr,coluna_legenda): # Coluna_legenda é a lista onde todas as subexpressões evariáveis são armazenadas
    
    pilha = [] # A pilha é a lista que guarda as posições dos parênteses de abertura

    for i in range(len(expr)): # i é o índice do caractere atual
        if expr[i] == "(":
            pilha.append(i) # Guarda o índice na pilha, e continu aprocurando por outros parênteses do tipo "("

        elif expr[i] == ")": #  Retira o último parêntese da pilha para evitar experessões inválidas, como "((A&B)"
            inicio = pilha.pop() # Inicio é o índice (a posição) do parêntese de abertura que acabou de ser encontrado

            # Verifica se esse par de parênteses engloba a expressão inteira. Se sim, se ignora este 
            # bloco e salta para o próximo caractere, pois a expressão inteira não é uma subexpressão.
            if inicio == 0 and i == len(expr) - 1:
                continue

            sub = expr[inicio:i + 1] # Extrai a subexpressão a ser guardada, com i + 1 para incluir o parêntese de fechamento ')'
            legenda = remover_parenteses_externos(sub) # Rempove os parênteses externos da subexpressão específica 

            # Evita duplicação de subexpressões 
            if legenda not in coluna_legenda:
                coluna_legenda.append(legenda) # Adiciona a subexpressão à lista "coluna_legenda"
                encontrar_relacoes(sub,coluna_legenda) # Procura subexpressões dentro dela mesma de forma recursiva


def numero_variaveis_linhas(coluna_legenda):
    # Conta variáveis puras e define o tamanho (que é o número de lihas) da tabela 
    variaveis = 0

    for elemento in coluna_legenda:
        if len(elemento) == 1 and elemento.isalpha(): # Se o elemento tem só um caractere, que é alfabético,
            variaveis += 1                            # então ele é considerado uma variável pura
            
    return variaveis, 2 ** variaveis
    # Retorna o total de variáveis encontradas e o número de linhas da tabela, 
    # calculado pela fórmula matemática 2^n, onde n é o número de variáveis 


def gerar_combinacoes_bool(n, tabela):
    # Gera uma matriz com todas as combinações possíveis de True/False

    for i in range(2 ** n): # Um loop no número exato de linhas da tabela
        linha = [] # A cada linha, se cria uma nova lista vazia

        for j in range(n - 1, -1, -1): # Um loop que percorre as colunas de trás para frente
            linha.append(bool((i >> j) & 1)) # Operação de Bitwise:
    # >> j : move o bit da coluna atual para a ponta direita
    # & 1 : isola esse bit (zera os outros) para descobrir se ele é 0 (False) ou 1 (True)

        tabela.append(linha)

    return tabela # Retorna a tabela preenchida coom a mtriz de valores booleanos

def gerar_bool_subexpressões(coluna_legenda, tabela):
    # avalia todas as subexpressões na tabela

    variaveis = [variavel for variavel in coluna_legenda if len(variavel) == 1]  # Da lista de subexpressões, seleciona os caracteres
                                                                                 # que são variáveis (como A, B, C, etc)
    idx = {nome: i for i, nome in enumerate(coluna_legenda)} # Dicionário que mapeia cada nome (variável ou subexpressão)             
                                                             # ao seu índice de coluna na tabela

    for linha in tabela[1:]: # Passa por todas as linhas de dados da tabela, pulando a primeira linha de cabeçalho

        while len(linha) < len(coluna_legenda):
            linha.append(None) # Adiciona Valor Vazio para abrir espaço nas colunas das 
                               # subexpressões que serão calculadas logo a seguir 

    for i, linha in enumerate(tabela): # i é o índice numérico e "linha" é a lista com os dados dessa linha

        if i == 0: # Novamente, pula a primeira linha de cabeçalho, que só contém texto
            continue

        for coluna in coluna_legenda: # Percore os valores (subexpressões e variáveis) de cada coluna

            if coluna in variaveis: # Se for variável, salta para a próxima coluna, pois todas as variáveis
                                 # já possuem valores boleanos definidos na matriz de combinações
                continue

            expr = coluna # expr recebe a subexpressão da coluna atual

            for variavel in variaveis:
                # [CORREÇÃO BUG 1]: Adiciona espaços para proteger palavras reservadas como 'and'
                valor_com_espaco = f" {str(linha[idx[variavel]])} "
                expr = expr.replace(variavel, valor_com_espaco)

            # Traduz operadores lógicos para Python:

            expr = expr.replace("<->", " == ")
            # [CORREÇÃO BUG 2]: 
            if "->" in expr:
                expr = "not " + expr.replace("->", ") or (")
            else:
                expr = expr.replace("->", ") or (")
            expr = expr.replace("&", " and ")
            expr = expr.replace("|", " or ")
            expr = expr.replace("~", " not ")

            # Envolve a expressão já traduzida em parênteses, para garantir a ordem matemática
            expr = "(" + expr + ")"

            linha[idx[coluna]] = eval(expr)
            # eval(expr) : Executa a operação booleana
            # linha[idx[coluna]] : Guarda o resultado na posição da matriz antes com valor vazio (None)

    return tabela # Retorna a matriz totalmente preenchida


def programa():
    # ============================================================
    # ENTRADA
    # ============================================================
    expressao = input("Digite a expressão lógica: ").replace(" ", "") # Solicita Input ao usuário e retira os espaços vazios

    # Validação manual de caracteres inválidos:
    contador_invalidos = 0
    for caracter in expressao:
        caracteres = [
        'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
        'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
        'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
        'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z','(',')','&','|','<','>','-','~'
    ]
        if caracter not in caracteres:
            contador_invalidos+=1
            break
    if contador_invalidos >= 1:
        print('Expressão Inválida (Presença de caracteres indevidos)')
    else: # Se não houver nenhum caracter inválido:

        pilha = [] # Pilha é a lista com os parênteses de abertura encontrados, para verificar se serão fechados
        erro = False

        for caracter in expressao:
            if caracter == "(": # Adiciona o parêntese de abertura na Pilha
                pilha.append(caracter)

            elif caracter == ")":
                if not pilha: # Verifica se a Pilha está vazia, se sim, significa que há um parêntese de 
                              # fechamento sem um correspondente de abertura, o que é um erro de sintaxe
                    erro = True
                    break
                else: # Se a pilha não estiver vazia, significa que o parêntese que foi aberto antes foi fechado
                      # Por isso, o retira da Pilha, zerando a pendência de fechamento
                    pilha.pop()

        if erro or pilha: # Se houver um erro de sintaxe ou se a Pilha não estiver vazia,
                          # significa que há parênteses de abertura sem fechamento
            print("Erro: parênteses inválidos.")

        else:
            expressao = remover_parenteses_externos(expressao) # Remove redundâncias antes de validar o envelopamento
            if expressao[0] != "(" or expressao[-1] != ")":
            # Se faltar um parêntese no início ou no fim, o código "envelopa" a 
            # expressão inteira dentro de um novo par de parênteses:
                expressao = "(" + expressao + ")"
            
            expressao = remover_parenteses_externos(expressao)
            # Testa novamente se os parênteses das pontas são redundantes, e os remove se for o caso

            # ========================================================
            # CONSTRUÇÃO DO CABEÇALHO (COLUNAS DA TABELA)
            # ========================================================
            
            coluna_legenda = [] # Recria a lista para armanezar

            # Procura por Variáveis na expressão e, se não estiver na lista, a adiciona
            for caracter in expressao:
                if caracter.isalpha() and caracter not in coluna_legenda:
                    coluna_legenda.append(caracter)

            # Para negações explícitas:
            for i, caracter in enumerate(expressao): # Mapeia os índices dos caracetres
                if caracter.isalpha(): # Se for variável, verifica se o anterior é uma negação "~"

                    if i > 0 and expressao[i - 1] == "~": # Se não é o primeiro e tem a negação antes
                        token = f"~{caracter}" # Cria um token para a variável negada, como "~A"

                        if token not in coluna_legenda: # Põe a variável com negação na lista
                            coluna_legenda.append(token)

            # Procura novamente por subexpressões internas
            encontrar_relacoes(expressao,coluna_legenda)

            # Verifica se a expressão completa insrida já foi listada no passo anterior
            if expressao not in coluna_legenda:
                coluna_legenda.append(expressao)

            # Remove duplicatas na lista
            coluna_legenda = list(dict.fromkeys(coluna_legenda))

            # ============================================================
            # GERAÇÃO DA TABELA VERDADE E CLASSIFICAÇÃO AUTOMÁTICA
            # ============================================================

            variaveis, _ = numero_variaveis_linhas(coluna_legenda) # Função para achar variáveis na lista
            # O underline (_) é usado para descartar o segundo valor retornado (que é o número de linhas),
            # já que o programa calcula isso novamente depois

            tabela = [coluna_legenda] # Inicia a matriz tabela com o cabeçalho na priemira linha

            tabela = gerar_combinacoes_bool(variaveis, tabela) # Função com os valores booleanos nas linhas na tabela

            tabela = gerar_bool_subexpressões(coluna_legenda, tabela) # Função para calcular as subexpressões e preencher a tabela

            contador_true = 0 # Contador de linhas com resultado True

            for i in range(1, len(tabela)): # Loop nas linhas da Tabela Verdade, ignorando o cabeçalho
                if tabela[i][-1] == True: # Na última coluna da linha atual, verifica se é True
                    contador_true += 1

            if contador_true == len(tabela[1:]): # Se o total de True é igual ao total de linhas:
                print('A expressão lógica é uma Tautologia')

            elif contador_true == 0: # Se nenhuma linha for True:
                print('A expressão lógica é uma Contradição')

            else: # Se houver tanto True como False:
                print('A expressão lógica é uma Contingência')
            


            df = pd.DataFrame(tabela[1:], columns=tabela[0])  # Biblioteca Pandas para organizar a Tabela
            display(df) # Exibe e retorna a Tabela de forma legível
            return df
        
def comparar_tabelas():
    tabela1 = programa() # Calcula a primeira Tabela
    tabela2 = programa() # Calcula a segunda Tabela

    if len(tabela1) == len(tabela2): # Se tiverem o mesmo número de linhas:
        if tabela1.iloc[:, -1].equals(tabela2.iloc[:, -1]): # Pandas verifica a última coluna da matriz
            print('As expressões são equivalentes') # Se tiverem os mesmos valores booleanos, são iguais

        else:
            print('Não são equivalentes') # Resultados booleanos diferentes
    else:
        print('Não são equivalentes') # Número de linhas diferente
        
def escolher():
    opção=input('O que você quer fazer?(Digite 1 para gerar 1 tabela verdade, Digite 2 para comparar duas tabelas)')
    if opção == '1':
        programa() 
    elif opção == '2':
        comparar_tabelas()
    else:
        print('Selecione uma opçaõ válida')
        escolher()      
escolher()